In [10]:
"""
完整158個技術指標計算程式
符合QLIB Alpha158標準
"""

import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

def safe_divide(numerator, denominator, fill_value=0):
    """安全除法，避免除以零"""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = np.where(
            np.abs(denominator) < 1e-10,
            fill_value,
            numerator / denominator
        )
        result = np.where(np.isfinite(result), result, fill_value)
    return result

def load_taiwan_stock_csv(filepath):
    """讀取台股CSV並處理"""
    print(f"\n{'='*70}")
    print(f"讀取檔案：{os.path.basename(filepath)}")
    print(f"{'='*70}")
    
    # 嘗試不同編碼
    for encoding in ['utf-8', 'utf-8-sig', 'big5', 'cp950', 'gbk']:
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            print(f"✓ 使用編碼：{encoding}")
            break
        except:
            continue
    else:
        raise ValueError("無法讀取檔案")
    
    # 重命名欄位
    if len(df.columns) >= 9:
        df.columns = ['Code', 'Name', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Amount']
    
    # 轉換數據類型
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Amount']
    for col in numeric_cols:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '').str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 轉換日期
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    
    # 移除缺失值並排序
    df = df.dropna().sort_values('Date').reset_index(drop=True)
    
    print(f"✓ 股票：{df['Name'].iloc[0]}")
    print(f"✓ 筆數：{len(df)}")
    print(f"✓ 日期：{df['Date'].min().date()} ~ {df['Date'].max().date()}")
    
    return df

def calculate_alpha158_features(df):
    """
    計算完整Alpha158特徵集
    結構：13個基礎 + 145個週期指標 = 158個
    """
    print(f"\n{'─'*70}")
    print(f"計算Alpha158技術指標...")
    print(f"{'─'*70}")
    
    features = pd.DataFrame(index=df.index)
    
    # 轉換為numpy數組加速計算
    close = df['Close'].values.astype(float)
    open_price = df['Open'].values.astype(float)
    high = df['High'].values.astype(float)
    low = df['Low'].values.astype(float)
    volume = df['Volume'].values.astype(float)
    
    # ============================================================
    # 第一組：原始價格特徵（5個）
    # ============================================================
    features['OPEN'] = open_price
    features['HIGH'] = high
    features['LOW'] = low
    features['CLOSE'] = close
    features['VOLUME'] = volume
    
    # ============================================================
    # 第二組：派生基礎特徵（8個）
    # ============================================================
    
    # VWAP - 成交量加權平均價
    typical_price = (high + low + close) / 3
    cumulative_tp_volume = np.cumsum(typical_price * volume)
    cumulative_volume = np.cumsum(volume)
    features['VWAP'] = safe_divide(cumulative_tp_volume, cumulative_volume, close[0])
    
    # K線形態特徵（7個）
    features['KMID'] = (close + open_price) / 2
    features['KMID2'] = (2 * close - high - low) / 2
    features['KLEN'] = high - low
    features['KLOW'] = safe_divide(low - open_price, open_price, 0)
    features['KLOW2'] = safe_divide(low - close, close, 0)
    features['KTOP'] = safe_divide(high - open_price, open_price, 0)
    features['KTOP2'] = safe_divide(high - close, close, 0)
    
    print(f"✓ 基礎特徵：13個")
    
    # ============================================================
    # 第三組：多週期指標（145個 = 29×5）
    # ============================================================
    periods = [5, 10, 20, 30, 60]
    close_series = pd.Series(close)
    high_series = pd.Series(high)
    low_series = pd.Series(low)
    volume_series = pd.Series(volume)
    
    for idx, period in enumerate(periods, 1):
        print(f"計算週期 {period:2d}...", end=' ')
        
        # 1. MA - 移動平均
        features[f'MA_{period}'] = close_series.rolling(period, min_periods=1).mean()
        
        # 2. STD - 標準差
        features[f'STD_{period}'] = close_series.rolling(period, min_periods=1).std().fillna(0)
        
        # 3-4. MAX/MIN
        features[f'MAX_{period}'] = high_series.rolling(period, min_periods=1).max()
        features[f'MIN_{period}'] = low_series.rolling(period, min_periods=1).min()
        
        # 5. ROC - 變動率
        features[f'ROC_{period}'] = close_series.pct_change(period).fillna(0)
        
        # 6. RSV - 未成熟隨機值
        low_min = low_series.rolling(period, min_periods=1).min()
        high_max = high_series.rolling(period, min_periods=1).max()
        features[f'RSV_{period}'] = safe_divide(
            (close_series - low_min).values * 100,
            (high_max - low_min).values,
            50
        )
        
        # 7. RANK - 排名
        features[f'RANK_{period}'] = close_series.rolling(period, min_periods=1).apply(
            lambda x: (x.rank(method='average').iloc[-1] - 1) / (len(x) - 1) if len(x) > 1 else 0.5,
            raw=False
        )
        
        # 8. RESI - 線性回歸殘差
        def calc_resi(x):
            if len(x) < 2:
                return 0
            try:
                idx = np.arange(len(x))
                coef = np.polyfit(idx, x, 1)
                return x.iloc[-1] - (coef[0] * idx[-1] + coef[1])
            except:
                return 0
        
        features[f'RESI_{period}'] = close_series.rolling(period, min_periods=2).apply(calc_resi, raw=False).fillna(0)
        
        # 9. RSQR - R²值
        def calc_rsqr(x):
            if len(x) < 2:
                return 0
            try:
                idx = np.arange(len(x))
                corr = np.corrcoef(idx, x)[0, 1]
                return corr ** 2 if np.isfinite(corr) else 0
            except:
                return 0
        
        features[f'RSQR_{period}'] = close_series.rolling(period, min_periods=2).apply(calc_rsqr, raw=False).fillna(0)
        
        # 10-11. QTLU/QTLD - 分位數
        features[f'QTLU_{period}'] = close_series.rolling(period, min_periods=1).quantile(0.8)
        features[f'QTLD_{period}'] = close_series.rolling(period, min_periods=1).quantile(0.2)
        
        # 12-13. IMAX/IMIN - 極值位置
        features[f'IMAX_{period}'] = high_series.rolling(period, min_periods=1).apply(
            lambda x: x.argmax() / len(x) if len(x) > 0 else 0.5,
            raw=False
        ).fillna(0.5)
        
        features[f'IMIN_{period}'] = low_series.rolling(period, min_periods=1).apply(
            lambda x: x.argmin() / len(x) if len(x) > 0 else 0.5,
            raw=False
        ).fillna(0.5)
        
        # 14. IMXD - 極值時間差
        features[f'IMXD_{period}'] = features[f'IMAX_{period}'] - features[f'IMIN_{period}']
        
        # 15-16. CORR/CORD - 相關係數
        features[f'CORR_{period}'] = close_series.rolling(period, min_periods=2).corr(volume_series).fillna(0)
        features[f'CORD_{period}'] = close_series.diff().rolling(period, min_periods=2).corr(volume_series.diff()).fillna(0)
        
        # 17-19. 漲跌天數統計
        price_change = close_series.diff()
        features[f'CNTP_{period}'] = (price_change > 0).rolling(period, min_periods=1).sum()
        features[f'CNTN_{period}'] = (price_change < 0).rolling(period, min_periods=1).sum()
        features[f'CNTD_{period}'] = (price_change == 0).rolling(period, min_periods=1).sum()
        
        # 20-22. 漲跌幅度累積
        features[f'SUMP_{period}'] = price_change.clip(lower=0).rolling(period, min_periods=1).sum()
        features[f'SUMN_{period}'] = price_change.clip(upper=0).abs().rolling(period, min_periods=1).sum()
        features[f'SUMD_{period}'] = price_change.abs().rolling(period, min_periods=1).sum()
        
        # 23-24. 成交量均線
        features[f'VMA_{period}'] = volume_series.rolling(period, min_periods=1).mean()
        features[f'VSTD_{period}'] = volume_series.rolling(period, min_periods=1).std().fillna(0)
        
        # 25-27. 成交量變動
        volume_change = volume_series.diff()
        features[f'VSUMP_{period}'] = volume_change.clip(lower=0).rolling(period, min_periods=1).sum()
        features[f'VSUMN_{period}'] = volume_change.clip(upper=0).abs().rolling(period, min_periods=1).sum()
        features[f'VSUMD_{period}'] = volume_change.abs().rolling(period, min_periods=1).sum()
        
        # 28. WVMA - 加權成交量均線
        def calc_wma(x):
            if len(x) == 0:
                return 0
            weights = np.arange(1, len(x) + 1)
            return np.sum(x * weights) / np.sum(weights)
        
        features[f'WVMA_{period}'] = volume_series.rolling(period, min_periods=1).apply(calc_wma, raw=False)
        
        # 29. BETA - 波動率比
        returns = close_series.pct_change().fillna(0)
        mean_ret = returns.rolling(period, min_periods=1).mean()
        std_ret = returns.rolling(period, min_periods=2).std().fillna(0)
        features[f'BETA_{period}'] = safe_divide(
            std_ret.values,
            np.abs(mean_ret.values),
            1.0
        )
        
        print(f"✓ ({idx}/5 完成，累計 {13 + idx * 29} 個)")
    
    # 最終處理
    print(f"\n處理異常值...", end=' ')
    features = features.fillna(method='ffill').fillna(method='bfill').fillna(0)
    features = features.replace([np.inf, -np.inf], 0)
    print("✓")
    
    print(f"\n{'─'*70}")
    print(f"✅ 特徵計算完成：{features.shape[1]} 個特徵")
    print(f"{'─'*70}")
    
    return features

def prepare_training_dataset(csv_path, predict_days=11):
    """
    準備訓練數據集
    predict_days: 預測未來第幾天的收盤價（預設11天）
    """
    print(f"\n{'='*70}")
    print(f"準備訓練數據集（預測 {predict_days} 天後收盤價）")
    print(f"{'='*70}")
    
    # 讀取原始數據
    df = load_taiwan_stock_csv(csv_path)
    dates = df['Date'].copy()
    
    # 計算158個特徵
    features = calculate_alpha158_features(df)
    
    # 添加日期
    features.insert(0, 'Date', dates.values)
    
    # 創建目標變量（未來N天的收盤價）
    features['Close'] = df['Close'].shift(-predict_days)
    
    # 移除沒有目標值的行
    valid_rows = len(features) - predict_days
    features = features.iloc[:valid_rows]
    
    print(f"\n✅ 數據集準備完成")
    print(f"   樣本數：{features.shape[0]} 筆")
    print(f"   特徵數：{features.shape[1] - 2} 個")
    print(f"   預測目標：未來第 {predict_days} 個交易日的收盤價")
    print(f"{'='*70}\n")
    
    return features

# ============================================================
# 主程序：批量處理
# ============================================================
if __name__ == '__main__':
    import time
    
    print("="*70)
    print("🚀 台股Alpha158特徵計算系統")
    print("="*70)
    
    # 配置
    stock_files = {
        '台積電': '台積電.csv',
        '長榮': '長榮.csv',
        '聯發科': '聯發科.csv',
        '統一超': '統一超.csv'
    }
    
    predict_days = 11  # ⭐ 可修改：預測天數
    output_dir = 'D:/pythonProject/a-lunwen/data'
    os.makedirs(output_dir, exist_ok=True)
    
    total_start = time.time()
    success_count = 0
    
    for idx, (stock_name, filename) in enumerate(stock_files.items(), 1):
        print(f"\n{'#'*70}")
        print(f"# [{idx}/{len(stock_files)}] {stock_name}")
        print(f"{'#'*70}")
        
        try:
            dataset = prepare_training_dataset(filename, predict_days)
            
            output_path = f'{output_dir}/{stock_name}_date.csv'
            dataset.to_csv(output_path, index=False, encoding='utf-8-sig')
            
            print(f"✅ 成功！位置：{output_path}")
            success_count += 1
            
        except Exception as e:
            print(f"❌ 失敗：{e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*70}")
    print(f"✅ 完成：{success_count}/{len(stock_files)}")
    print(f"⏱️  耗時：{time.time() - total_start:.1f} 秒")
    print(f"{'='*70}")

🚀 台股Alpha158特徵計算系統

######################################################################
# [1/4] 台積電
######################################################################

準備訓練數據集（預測 11 天後收盤價）

讀取檔案：台積電.csv
✓ 使用編碼：big5
✓ 股票：台積電
✓ 筆數：1152
✓ 日期：2021-01-04 ~ 2025-09-30

──────────────────────────────────────────────────────────────────────
計算Alpha158技術指標...
──────────────────────────────────────────────────────────────────────
✓ 基礎特徵：13個
計算週期  5... ✓ (1/5 完成，累計 42 個)
計算週期 10... ✓ (2/5 完成，累計 71 個)
計算週期 20... ✓ (3/5 完成，累計 100 個)
計算週期 30... ✓ (4/5 完成，累計 129 個)
計算週期 60... ✓ (5/5 完成，累計 158 個)

處理異常值... ✓

──────────────────────────────────────────────────────────────────────
✅ 特徵計算完成：158 個特徵
──────────────────────────────────────────────────────────────────────

✅ 數據集準備完成
   樣本數：1141 筆
   特徵數：158 個
   預測目標：未來第 11 個交易日的收盤價

✅ 成功！位置：D:/pythonProject/a-lunwen/data/台積電_date.csv

######################################################################
# [2/4] 長榮
###################################

In [ ]:
"""
台股HFSLS-PSO-BIGRU預測系統 - 完整版
支援Alpha158特徵集（158個特徵）
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 20          # 滑動窗口大小
loss = "mse"
sample_ratios = [1, 0.9, 0.8]
bins_fs = 3
enable_fs = True
sub_features = []
ensemble = []
num_models = 3       # 模型層數
minloss = 20
s = 0
measure = [[]]
true_pre = [[], []]

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_layers, total_stocks):
        self.total_layers = total_layers
        self.total_stocks = total_stocks
        self.start_time = time.time()
        self.layer_times = []
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f"📈 [{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
        self.layer_times = []
    
    def start_layer(self, layer_num):
        self.layer_start = time.time()
        print(f"\n{'─'*70}")
        print(f"🔄 第 {layer_num}/{self.total_layers} 層訓練")
        print(f"{'─'*70}")
    
    def pso_progress(self, iteration, total, best_mse):
        if iteration % 5 == 0 or iteration == total:
            percent = (iteration / total) * 100
            filled = int(30 * iteration / total)
            bar = '█' * filled + '░' * (30 - filled)
            print(f"\r  PSO: |{bar}| {percent:>5.1f}% | MSE: {best_mse:.6f}", end='')
    
    def end_layer(self, layer_num, feature_count):
        layer_time = time.time() - self.layer_start
        self.layer_times.append(layer_time)
        avg_time = np.mean(self.layer_times)
        remaining = self.total_layers - layer_num
        eta = timedelta(seconds=int(avg_time * remaining))
        
        print(f"\n  ✓ 完成：{layer_time/60:.1f}分鐘")
        print(f"  ✓ 特徵：{feature_count}個")
        print(f"  ⏱  預計剩餘：{eta}")
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f" {stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def weight(n, k):
    w = np.arange(1, k * n + 1, k)
    return w / w.sum()

def process_data(X):
    """轉換為滑動窗口格式"""
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_bigru_model(params, input_shape):
    """構建BIGRU模型"""
    units, dropout_rate, _ = params
    model = Sequential([
        Bidirectional(GRU(int(units), activation='relu', return_sequences=False), 
                     input_shape=input_shape),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def fitness_function(params, x_train, y_train, x_val, y_val):
    """PSO適應度函數"""
    try:
        model = build_bigru_model(params, x_train.shape[1:])
        model.fit(x_train, y_train, batch_size=32, epochs=8, verbose=0, shuffle=False)
        y_pred = model.predict(x_val, verbose=0)
        return mean_squared_error(y_val, y_pred)
    except:
        return float('inf')

def PSO(x_train, y_train, x_val, y_val, pop_size=8, max_iter=25):
    """粒子群優化"""
    global tracker
    
    param_bounds = np.array([[32, 96], [0.1, 0.4], [0.0005, 0.005]])
    population = np.random.uniform(param_bounds[:, 0], param_bounds[:, 1], (pop_size, 3))
    velocities = np.random.uniform(-1, 1, (pop_size, 3))
    
    pbest_pos = population.copy()
    pbest_score = np.full(pop_size, float('inf'))
    gbest_pos = np.zeros(3)
    gbest_score = float('inf')
    
    for t in range(max_iter):
        for i in range(pop_size):
            fitness = fitness_function(population[i], x_train, y_train, x_val, y_val)
            
            if fitness < pbest_score[i]:
                pbest_score[i] = fitness
                pbest_pos[i] = population[i]
            
            if fitness < gbest_score:
                gbest_score = fitness
                gbest_pos = population[i]
        
        w, c1, c2 = 0.5, 1.5, 1.5
        for i in range(pop_size):
            r1, r2 = np.random.rand(2)
            velocities[i] = (w * velocities[i] +
                           c1 * r1 * (pbest_pos[i] - population[i]) +
                           c2 * r2 * (gbest_pos - population[i]))
            population[i] = np.clip(population[i] + velocities[i], 
                                   param_bounds[:, 0], param_bounds[:, 1])
        
        if tracker:
            tracker.pso_progress(t+1, max_iter, gbest_score)
    
    print(f"\n  ✓ 最佳參數：units={int(gbest_pos[0])}, dropout={gbest_pos[1]:.3f}")
    return gbest_pos

def train_submodel(x_train, y_train, x_test, y_test, test_indices, dates, stock_name):
    """訓練子模型"""
    global minloss
    
    # PSO優化
    print(f"   PSO優化中...")
    best_params = PSO(x_train, y_train, x_test, y_test)
    
    # 構建最終模型
    print(f"    構建模型...")
    model = Sequential([
        Bidirectional(GRU(int(best_params[0]), activation='relu', return_sequences=False),
                     input_shape=x_train.shape[1:]),
        Dropout(best_params[1]),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss=loss)
    model.fit(x_train, y_train, batch_size=32, epochs=30, 
             validation_split=0.1, verbose=0, shuffle=False)
    
    # 預測與評估
    y_pred = model.predict(x_test, verbose=0)
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"   MSE={mse:.6f}, R²={r2:.4f}")
    
    # 保存最佳結果
    if mse < minloss:
        y_pred_inv = scaler.inverse_transform(y_pred)
        y_test_inv = scaler.inverse_transform(y_test)
        
        measure[0] = [mse, rmse, mae, mape, r2]
        true_pre[0] = list(y_test_inv[:, 0])
        true_pre[1] = list(y_pred_inv[:, 0])
        minloss = mse
        
        # 保存預測結果（含日期）
        base_dates = [dates[idx] for idx in test_indices]
        predict_dates = [d + pd.Timedelta(days=15) for d in base_dates]
        
        data = pd.DataFrame({
            'stock_name': stock_name,
            'base_date': base_dates,
            'predict_date': predict_dates,
            'predicted_close': y_pred_inv[:, 0],
            'actual_close': y_test_inv[:, 0],
            'error': y_pred_inv[:, 0] - y_test_inv[:, 0],
            'error_pct': ((y_pred_inv[:, 0] - y_test_inv[:, 0]) / y_test_inv[:, 0] * 100)
        })
        
        output_path = 'D:/pythonProject/a-lunwen/data/predicted_sse.csv'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        try:
            existing = pd.read_csv(output_path)
            existing = pd.concat([existing, data], ignore_index=True)
        except FileNotFoundError:
            existing = data
        
        existing.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"   已保存")
    
    return model

def predict_sub(model, x_train):
    return pd.Series(model.predict(x_train, verbose=0).squeeze())

def get_loss(label, pred):
    return (np.squeeze(label) - pred) ** 2

def feature_selection(X, y, y_train, N, loss_values, features, w):
    """HFSLS特徵選擇"""
    print(f"   特徵選擇（{len(features)}個）...", end='')
    
    F = len(features)
    E = pd.DataFrame({"E_value": np.zeros(F)})
    X_tmp = X.copy()
    
    for i_f, feat in enumerate(features):
        X_tmp.loc[:, feat] = np.random.permutation(X_tmp.loc[:, feat].values)
        pred = pd.Series(np.zeros(N))
        
        for i_s, model in enumerate(ensemble):
            x = process_data(X_tmp.loc[:, sub_features[i_s]].values)
            x_train, x_test, _, _ = train_test_split(x, y, test_size=0.3, shuffle=True)
            pred += predict_sub(model, x_train) * w[i_s]
        
        loss_feat = get_loss(y_train, pred.values)
        E.loc[i_f, "E_value"] = np.mean(loss_feat - loss_values) / (np.std(loss_feat - loss_values) + 1e-7)
        X_tmp.loc[:, feat] = X.loc[:, feat]
    
    E["E_value"].fillna(0, inplace=True)
    E["bins"] = pd.cut(E["E_value"], bins_fs)
    
    res_feat = []
    for i_b, b in enumerate(sorted(E["bins"].unique(), reverse=True)):
        b_feat = features[E["bins"] == b]
        ratio = sample_ratios[i_b] if i_b < len(sample_ratios) else 0.5
        num_feat = int(np.ceil(ratio * len(b_feat)))
        
        if num_feat > 0 and len(b_feat) > 0:
            selected = np.random.choice(b_feat, min(num_feat, len(b_feat)), replace=False)
            res_feat.extend(selected)
    
    print(f" → {len(res_feat)}個 ✓")
    return pd.Index(set(res_feat))

def fit(df, dates, stock_name):
    """主訓練函數"""
    global measure, true_pre, minloss, sub_features, ensemble, tracker
    
    print(f"\n 數據：{df.shape[0]}樣本 × {df.shape[1]}特徵")
    
    # 歸一化特徵
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    features = X.columns
    
    # 準備標籤
    y = df['Close'].values[:-window+1]
    y = scaler.fit_transform(y.reshape(-1, 1))
    valid_dates = dates[:-window+1].reset_index(drop=True)
    
    pred_sub = pd.DataFrame(np.zeros((int(len(y) * 0.7), num_models)))
    
    # 分層訓練
    for k in range(num_models):
        if tracker:
            tracker.start_layer(k+1)
        
        sub_features.append(features)
        x = process_data(X.loc[:, features].values)
        
        x_train, x_test, y_train, y_test, train_idx, test_idx = train_test_split(
            x, y, range(len(x)), test_size=0.3, shuffle=True, random_state=42
        )
        
        split_ = int(len(y) * 0.7)
        x_test_final = x[split_:]
        y_test_final = y[split_:]
        test_indices = list(range(split_, len(y)))
        
        model_k = train_submodel(x_train, y_train, x_test_final, y_test_final,
                                test_indices, valid_dates, stock_name)
        ensemble.append(model_k)
        
        if tracker:
            tracker.end_layer(k+1, len(features))
        
        if k + 1 == num_models:
            break
        
        # 預測與特徵選擇
        pred_k = predict_sub(model_k, x_train)
        pred_sub.iloc[:, k] = pred_k
        
        if s == 0:
            pred_ensemble = pred_sub.iloc[:, :k+1].mean(axis=1)
            w = np.ones(k+1) / (k+1)
        else:
            w = weight(k+1, s)
            pred_ensemble = (pred_sub.iloc[:, :k+1] * w).sum(axis=1)
        
        loss_values = pd.Series(get_loss(y_train, pred_ensemble.values))
        
        if enable_fs and k < num_models - 1:
            features = feature_selection(X, y, y_train, len(x_train), loss_values, features, w)

def main(data, stock_name):
    """主函數"""
    global minloss, measure, true_pre, sub_features, ensemble
    
    # 重置全局變量
    minloss, measure, true_pre = 20, [[]], [[], []]
    sub_features, ensemble = [], []
    
    # 讀取數據
    df_full = pd.read_csv(f'D:/pythonProject/a-lunwen/data/{data}')
    
    if 'Date' in df_full.columns:
        dates = pd.to_datetime(df_full['Date'])
        df = df_full.drop(columns=['Date'])
    else:
        dates = pd.date_range('2021-01-01', periods=len(df_full), freq='D')
        df = df_full
    
    df = df.fillna(0).sort_index()
    
    # 訓練
    fit(df, dates, stock_name)
    
    # 輸出結果
    print(f"\n{'='*70}")
    print(f" {stock_name} 完成！")
    print(f"{'='*70}")
    print(f" 最終指標：")
    print(f"   MSE:  {measure[0][0]:.6f}")
    print(f"   RMSE: {measure[0][1]:.6f}")
    print(f"   MAE:  {measure[0][2]:.6f}")
    print(f"   MAPE: {measure[0][3]:.4f}")
    print(f"   R²:   {measure[0][4]:.4f}")
    print(f"{'='*70}\n")

# ============================================================
# 主程序
# ============================================================
if __name__ == '__main__':
    print("="*70)
    print(" 台股HFSLS-PSO-BIGRU預測系統")
    print("   Alpha158特徵集（158個特徵）")
    print("   預測目標：11個交易日後的收盤價")
    print("="*70)
    
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(num_models, len(stock_list))
    
    total_start = time.time()
    success = 0
    
    for idx, stock in enumerate(stock_list, 1):
        try:
            tracker.start_stock(stock, idx)
            main(f"{stock}_date.csv", stock)
            tracker.end_stock(stock)
            success += 1
        except Exception as e:
            print(f"\n❌ {stock} 失敗：{e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*70}")
    print(f" 全部完成！")
    print(f"{'='*70}")
    print(f" 成功：{success}/{len(stock_list)}")
    print(f"  耗時：{(time.time()-total_start)/60:.1f} 分鐘")
    print(f" 結果：D:/pythonProject/a-lunwen/data/predicted_sse.csv")
    print(f"{'='*70}")

 台股HFSLS-PSO-BIGRU預測系統
   Alpha158特徵集（158個特徵）
   預測目標：11個交易日後的收盤價

📈 [1/4] 處理：台積電

 數據：1141樣本 × 159特徵

──────────────────────────────────────────────────────────────────────
🔄 第 1/3 層訓練
──────────────────────────────────────────────────────────────────────
   PSO優化中...
  PSO: |██████████████████████████████| 100.0% | MSE: 0.000721
  ✓ 最佳參數：units=56, dropout=0.100
    構建模型...
   MSE=0.001204, R²=0.9205
   已保存

  ✓ 完成：36.7分鐘
  ✓ 特徵：158個
  ⏱  預計剩餘：1:13:18
   特徵選擇（158個）... → 143個 ✓

──────────────────────────────────────────────────────────────────────
🔄 第 2/3 層訓練
──────────────────────────────────────────────────────────────────────
   PSO優化中...
  PSO: |██████████████████████████████| 100.0% | MSE: 0.000771
  ✓ 最佳參數：units=89, dropout=0.100
    構建模型...
   MSE=0.001768, R²=0.8832

  ✓ 完成：48.8分鐘
  ✓ 特徵：143個
  ⏱  預計剩餘：0:42:44
   特徵選擇（143個）... → 127個 ✓

──────────────────────────────────────────────────────────────────────
🔄 第 3/3 層訓練
───────────────────────────────────────────────────────────

In [2]:
pip install tensorflow

INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/276.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/276.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/276.5 MB 2.6 MB/s eta 0:01:47
   ---------------------------------------- 1.0/276.5 MB 2.6 MB/s eta 0:01:45
   ---------------------------------------- 1.8/276.5 MB 2.7 MB/s eta 0:01:41
   ---------------------------------------- 2.6/276.5 MB 3.3 MB/s eta 0:01:24
   ---------------------------------------- 3.4/276.5 MB 3.1 MB/s eta 0:01:29
    --------------------------------------- 3.9/276.5 MB 3.1 MB/s eta 0:01:30
    --------------------------------------- 4.5/276.5 MB 3.1 MB/s eta 0:01:30
    --------------------------------------- 5.0/276.5 MB 3.0 MB/s eta 0:01:31
    --------------------------------------- 5.2/276.5 MB 2.9 MB/s eta 0:01:34
    -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
pip install pandas

^C
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-2.0.3-cp38-cp38-win_amd64.whl.metadata (18 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached numpy-1.24.4-cp38-cp38-win_amd64.whl.metadata (5.6 kB)
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
    --------------------------------------- 0.3/10.8 MB ? eta -:--:--
    --------------------------------------- 0.3/10.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.8 MB 882.6 kB/s eta 0:00:12
   - -------------------------------------- 0.5/10.8 MB 882.6 kB/s eta 0:00:12
   - -------------------------------------- 0.5/10.8 MB 882.6 kB/s eta 0:00:12
   -- ------------------------------------- 0.8/10.8 MB 493.2 kB/s eta 0:00:21
   -- ------------------------------------- 0.8/10.8 MB 493.2 k

ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\http\client.py", line 459, in read
    n = self.readinto(b)
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\http\client.py", line 503, in readinto
    n = self.fp.readinto(b)
  File "c:\Users\USER\anaconda3\envs\trex-game\lib\sock